# 1 Imports & Configuration

In [ ]:
import sys
import os
import time
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown, Image

# Add the 'src' directory to the system path to import modules
sys.path.append(os.path.abspath('src'))

# Import Project Modules
from config import Config
from preprocessor import TextPreprocessor
from textrank import TextRankSummarizer
from llm_engine import LLMSummarizer
from merger import HybridMerger

print("Loading Models... This might take a minute.")

# Initialize the engines once to save time during interaction
# 1. Initialize TextRank (Classical)
tr_engine = TextRankSummarizer()

# 2. Initialize LLM (Oracle) - Uses the model defined in Config
llm_engine = LLMSummarizer(Config.LLM_MODEL_NAME)

# 3. Initialize Merger
merger = HybridMerger(alpha=Config.ALPHA)

print(f"✅ System Ready! Loaded Model: {Config.LLM_MODEL_NAME}")
print(f"✅ Hybrid Alpha: {Config.ALPHA} (Weighting: {Config.ALPHA*100}% Graph, {(1-Config.ALPHA)*100}% Semantic)")

# 2 Interactive Interface

In [ ]:
# --- UI Layout ---

# 1. Title
header = widgets.HTML("<h2>📝 Hybrid Summarization Demo (Path 3)</h2>")

# 2. Input Text Area
input_text_area = widgets.Textarea(
    value='',
    placeholder='Paste your text here (News article, Technical paper, Wiki abstract)...',
    description='',
    layout=widgets.Layout(width='100%', height='200px')
)

# 3. Action Button
summarize_btn = widgets.Button(
    description='Generate Summary',
    button_style='primary', # 'success', 'info', 'warning', 'danger' or ''
    layout=widgets.Layout(width='200px'),
    icon='magic'
)

# 4. Output Container
output_area = widgets.Output()

# --- Logic Function ---
def on_click_summarize(b):
    with output_area:
        clear_output()
        text = input_text_area.value.strip()
        
        if not text:
            print("⚠️ Please enter some text first.")
            return
        
        print("⏳ Processing... Please wait.")
        start_time = time.time()
        
        try:
            # Step 1: Preprocessing
            sentences = TextPreprocessor.split_sentences(text)
            print(f"🔹 Detected {len(sentences)} sentences.")
            
            # Step 2: Classical TextRank
            tr_scores = tr_engine.rank_sentences(sentences)
            
            # Step 3: LLM Oracle
            oracle_summary = llm_engine.generate_oracle_summary(text)
            
            # Step 4: Hybrid Merge
            final_scores = merger.merge_scores(sentences, tr_scores, oracle_summary)
            final_summary = merger.get_top_k(sentences, final_scores, k=Config.TOP_K_SENTENCES)
            
            total_time = time.time() - start_time
            
            # --- Display Results ---
            clear_output()
            
            display(Markdown(f"### ⏱️ Execution Time: {total_time:.4f} seconds"))
            
            display(Markdown("---"))
            display(Markdown("### 🤖 Phase 2: LLM Oracle Summary (Abstractive)"))
            print(oracle_summary)
            
            display(Markdown("---"))
            display(Markdown("### 🏆 Final Hybrid Summary (Extractive)"))
            display(Markdown(f"> **{final_summary}**"))
            
            display(Markdown("---"))
            
        except Exception as e:
            print(f"❌ Error: {e}")

# Link button to function
summarize_btn.on_click(on_click_summarize)

# Display UI
display(header, input_text_area, summarize_btn, output_area)

# 3 Performance Analysis

In [ ]:
display(Markdown("## 📊 Empirical Performance Analysis"))
display(Markdown("The chart below demonstrates the **O(N²)** time complexity of the Hybrid TextRank algorithm, derived from the benchmark tests on the CPU."))

try:
    display(Image(filename='benchmarks/performance_plot.png'))
except FileNotFoundError:
    print("⚠️ Plot file not found. Please run 'benchmarks/analyze_performance.py' first.")

# 4 Configurations

In [ ]:
display(Markdown("### ⚙️ Current Configuration"))
print(f"Model: {Config.LLM_MODEL_NAME}")
print(f"Top K Sentences: {Config.TOP_K_SENTENCES}")
print(f"Alpha (Graph Weight): {Config.ALPHA}")
print(f"Similarity Threshold: {Config.SIMILARITY_THRESHOLD}")